# OASIS-1 MRI Dementia Classification

CNN that classifies brain MRI slices into `Non_Demented`, `Very_Mild_Demented`,
or `Mild_Demented`, using the OASIS-1 T88_111 masked_gfc dataset.

This notebook walks through the same pipeline as `src/train.py` step by
step, with inline visualizations. See `README.md` for the write-up of why
the training setup looks the way it does (balanced oversampling instead of
class-weighted loss, no BatchNorm, balanced-accuracy-based checkpointing) --
several simpler configurations collapsed to predicting a single class.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

from dataset import (
    load_dataset_info,
    load_labels,
    load_slices,
    load_split,
    class_names,
    balanced_train_dataset,
)
from model import build_model
from train import BalancedAccuracy, patient_level_predictions, BATCH_SIZE, SEED

tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__)

## Dataset overview

In [ ]:
info = load_dataset_info()
info

In [ ]:
labels_df = load_labels()
labels_df.head()

In [ ]:
counts = labels_df.groupby(["split", "class_name"]).size().unstack(fill_value=0)
counts = counts[info["classes"]]
counts.loc[["train", "val", "test"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts.loc[["train", "val", "test"]].plot(kind="bar", ax=ax)
ax.set_ylabel("slices")
ax.set_title("Class distribution per split")
plt.tight_layout()
plt.show()

## Sample slices

One example slice per class, so you can see what the model is actually
looking at.

In [ ]:
slices = load_slices()
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax, cls in zip(axes, info["classes"]):
    idx = labels_df.index[labels_df["class_name"] == cls][0]
    ax.imshow(slices[idx], cmap="gray")
    ax.set_title(cls)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Load the train/val/test splits

The split is by *patient* (not by slice), so there's no leakage between
sets -- every slice from a given patient is entirely in one split.

In [ ]:
X_train, y_train, patient_ids_train = load_split("train")
X_val, y_val, patient_ids_val = load_split("val")
X_test, y_test, patient_ids_test = load_split("test")

print("train:", X_train.shape, "patients:", len(set(patient_ids_train)))
print("val:  ", X_val.shape, "patients:", len(set(patient_ids_val)))
print("test: ", X_test.shape, "patients:", len(set(patient_ids_test)))

## Model architecture

In [ ]:
model = build_model(input_shape=X_train.shape[1:], num_classes=3)
model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4, clipnorm=1.0),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## Training

Training data is balanced by oversampling each class equally per batch
(`balanced_train_dataset`) rather than by loss reweighting -- a
`class_weight`-based loss repeatedly made the optimizer collapse to
predicting a single class early in training on this small, imbalanced
dataset.

Checkpointing/early-stopping monitor a custom `val_balanced_acc` metric
(mean per-class recall) instead of `val_loss`, since `val_loss` favored
epochs where the model leaned toward the majority class -- lowest
confidently-wrong penalty on the small validation split, but not the
epochs actually learning to recognize all three classes.

This will take a while to run (roughly 15-20 epochs x ~20s/epoch on CPU,
possibly up to 40 with early stopping).

In [ ]:
train_ds = balanced_train_dataset(X_train, y_train, BATCH_SIZE, SEED)
steps_per_epoch = len(X_train) // BATCH_SIZE

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

callbacks = [
    BalancedAccuracy(X_val, y_val),  # must run first to populate logs
    tf.keras.callbacks.EarlyStopping(
        monitor="val_balanced_acc", mode="max", patience=12, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_balanced_acc", mode="max", factor=0.5, patience=5, min_lr=1e-6
    ),
]

history = model.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    epochs=40,
    callbacks=callbacks,
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Raw accuracy")
axes[1].legend()

axes[2].plot(history.history["val_balanced_acc"], label="val", color="green")
axes[2].axhline(1 / 3, color="gray", linestyle="--", label="chance")
axes[2].set_title("Validation balanced accuracy")
axes[2].legend()

plt.tight_layout()
plt.show()

## Evaluation

Raw accuracy is misleading on this imbalanced test set (always predicting
`Non_Demented` alone gets ~58%), so look at per-class precision/recall.

In [ ]:
def plot_confusion(cm, labels, title):
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    fig.colorbar(im)
    plt.tight_layout()
    plt.show()

names = class_names()
test_probs = model.predict(X_test)
test_pred = test_probs.argmax(axis=1)

print(classification_report(y_test, test_pred, target_names=names))
plot_confusion(confusion_matrix(y_test, test_pred), names, "Per-slice (test)")

### Patient-level (majority vote)

The clinically meaningful unit: majority-voting a patient's slice
predictions into one label per patient.

In [ ]:
patient_true, patient_pred = patient_level_predictions(test_probs, patient_ids_test, y_test)

print(classification_report(patient_true, patient_pred, target_names=names))
plot_confusion(
    confusion_matrix(patient_true, patient_pred), names, "Patient-level majority vote (test)"
)

## Summary

- No more single-class collapse: the model discriminates all three classes.
- `Non_Demented` and `Mild_Demented` are recognized reasonably well;
  `Very_Mild_Demented` is the hardest class (it sits visually between the
  other two) and typically has the lowest recall.
- Expect some run-to-run variance in the exact numbers -- the test set is
  only 36 patients, and training is stochastic.
- This is a from-scratch 2D-slice CNN on a small dataset (163 unique
  training patients). Further gains would likely come from combining
  multiple slices per patient, adding the `age`/`mmse`/`nwbv` metadata as
  auxiliary features, or transfer learning from a pretrained medical
  imaging backbone -- not from more tuning of this architecture.